# ULA vs MALA

Chapter 18: Langevin Dynamics and Sampling · *Mathematics for AI and Machine Learning* by Fuheng Wu

Unadjusted vs Metropolis-adjusted Langevin on the same target.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-18/mala/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import math

import torch

torch.manual_seed(42)

# The demo's defaults: bimodal target, step eta, steps
steps, eta = 4000, 0.05


def log_target(x):
    # log(e^a + e^b) computed stably, so far-away proposals don't underflow to log(0)
    a, b = -0.5 * (x + 2) ** 2, -0.5 * (x - 2) ** 2
    hi = max(a, b)
    return hi + math.log(math.exp(a - hi) + math.exp(b - hi))


def score(x):
    a, b = math.exp(-0.5 * (x + 2) ** 2), math.exp(-0.5 * (x - 2) ** 2)
    return (a * (-(x + 2)) + b * (-(x - 2))) / (a + b)


def log_q(to, frm):
    """Log density of the Langevin proposal N(frm + eta * score(frm), 2 eta), up to a constant."""
    return -((to - frm - eta * score(frm)) ** 2) / (4 * eta)


def run(steps, eta, adjust, n_chains=1, x0=0.0):
    out, acc = [], 0
    x = x0
    for z, u in zip(torch.randn(steps).tolist(), torch.rand(steps).tolist()):
        y = x + eta * score(x) + math.sqrt(2 * eta) * z
        if not adjust:
            x = y                                          # ULA: always move
        else:
            # MALA: Metropolis-Hastings with the asymmetric proposal correction
            log_a = log_target(y) - log_target(x) + log_q(x, y) - log_q(y, x)
            if math.log(u) <= log_a:
                x, acc = y, acc + 1
        out.append(x)
    return torch.tensor(out), acc / steps


ula, _ = run(steps, eta, adjust=False)
mala, rate = run(steps, eta, adjust=True)
print(f"MALA acceptance rate: {rate:.3f}")
print(f"ULA  mean {ula.mean().item():+.3f}, variance {ula.var().item():.3f}")
print(f"MALA mean {mala.mean().item():+.3f}, variance {mala.var().item():.3f}  (target variance 5)")
# 4000 small steps is short for hopping between modes 4 apart, so both are still off.

# ── With a big step, ULA is biased; MALA's accept/reject removes the bias ───
big = 0.8
print(f"\nlarge step eta = {big}, 30000 steps each:")
eta = big
u_big, _ = run(30000, big, adjust=False)
m_big, r_big = run(30000, big, adjust=True)
print(f"ULA  variance {u_big.var().item():.3f}  (biased)")
print(f"MALA variance {m_big.var().item():.3f}  (target 5), acceptance {r_big:.3f}")

MALA acceptance rate: 0.997
ULA  mean +0.132, variance 4.221
MALA mean +0.561, variance 4.136  (target variance 5)

large step eta = 0.8, 30000 steps each:
ULA  variance 5.381  (biased)
MALA variance 5.006  (target 5), acceptance 0.845
